# Bayesian networks (directed models) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## A directed graph turns one large joint distribution into local conditional tables
Bayesian networks factor a joint distribution using parents. These examples use A -> B -> C to compute a joint table, marginals, conditional evidence, intervention contrast, and the parameter savings from factorization.

In [ ]:
# 1) joint table from P(A)P(B|A)P(C|B)
PA=np.array([0.4,0.6]); PB_A=np.array([[0.8,0.2],[0.3,0.7]]); PC_B=np.array([[0.9,0.1],[0.2,0.8]])
joint=np.zeros((2,2,2))
for a,b,c in itertools.product([0,1],repeat=3): joint[a,b,c]=PA[a]*PB_A[a,b]*PC_B[b,c]
plt.figure(figsize=(6,3)); plt.bar(range(8),joint.ravel()); plt.title('8 joint probabilities from 5 local numbers')
assert abs(joint.sum()-1)<1e-12 and abs(joint[1,1,1]-0.336)<1e-12

In [ ]:
# 2) marginal P(C=1) by summing hidden A and B
PC=joint.sum(axis=(0,1)); plt.figure(figsize=(6,3)); plt.bar(['C=0','C=1'],PC); plt.title(f'P(C=1)={PC[1]:.3f}')
assert abs(PC[1]-0.45)<1e-12

In [ ]:
# 3) evidence flows backward: P(A=1 | C=1)
postA=joint[:,:,1].sum(axis=1)/joint[:,:,1].sum(); plt.figure(figsize=(6,3)); plt.bar(['A=0','A=1'],postA); plt.title('observing C=1 raises belief in A=1')
assert abs(postA[1]-0.7866666666666666)<0.001

In [ ]:
# 4) intervention do(B=1) cuts incoming arrows to B
p_c_do_b1=PC_B[1,1]; observational=PC[1]
plt.figure(figsize=(6,3)); plt.bar(['observe average','do(B=1)'],[observational,p_c_do_b1]); plt.title('intervention uses child CPT directly')
assert abs(p_c_do_b1-0.8)<1e-12 and p_c_do_b1>observational

In [ ]:
# 5) parameter count: full joint vs directed factorization
full=2**3-1; fact=1+2+2
plt.figure(figsize=(6,3)); plt.bar(['full joint','BN factors'],[full,fact]); plt.title('factorization saves parameters')
assert full==7 and fact==5